# AlphaLens Point-in-Time Earnings Surprises

This notebook audits reported-versus-estimated EPS before the features enter a tree model. Yahoo calendar percentages are normalized to decimal rates, future estimate-only rows are excluded, and results can match only an earnings call within three calendar days when the result was observable by the feature anchor.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sqlalchemy import text

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.dataset import get_database_engine
from pipelines.ml.features import build_event_feature_dataset

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

## Source coverage

In [ ]:
engine = get_database_engine()
coverage = pd.read_sql_query(text('''
    SELECT ticker, COUNT(*) AS results,
           MIN(earnings_date) AS oldest, MAX(earnings_date) AS newest,
           COUNT(eps_surprise_pct) AS surprise_values
    FROM earnings_results
    GROUP BY ticker
    ORDER BY ticker
'''), engine)
display(coverage)

In [ ]:
results = pd.read_sql_query(text('''
    SELECT ticker, earnings_date, eps_estimate, reported_eps, eps_surprise_pct
    FROM earnings_results
    ORDER BY earnings_date, ticker
'''), engine)
results['outcome'] = results['eps_surprise_pct'].map(
    lambda value: 'Beat' if value > 0 else ('Miss' if value < 0 else 'Met')
)
display(results['outcome'].value_counts(dropna=False).rename('events'))
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(results['eps_surprise_pct'].clip(-0.5, 0.5), bins=35, ax=ax)
ax.axvline(0, color='#111827', linewidth=1)
ax.set(title='EPS surprise distribution (clipped to +/-50%)', xlabel='EPS surprise rate', ylabel='Events')
plt.tight_layout()
plt.show()

## Point-in-time call matches

SEC filing rows remain null by design. The table below measures how many earnings calls receive a bounded result match and whether provider dates differ from transcript dates.

In [ ]:
features = build_event_feature_dataset(horizon=30, include_topics=True)
calls = features.loc[features['event_source'] == 'earnings_call'].copy()
match_summary = pd.Series({
    'earnings_calls': len(calls),
    'matched_results': calls['reported_eps'].notna().sum(),
    'match_coverage': calls['reported_eps'].notna().mean(),
    'max_match_days': calls['earnings_result_match_days'].max(),
})
display(match_summary)
display(calls['earnings_result_match_days'].value_counts(dropna=False).sort_index())

In [ ]:
display(calls.loc[calls['reported_eps'].notna(), [
    'ticker', 'event_date', 'fiscal_period', 'earnings_result_date',
    'eps_estimate', 'reported_eps', 'eps_surprise_pct',
    'prior_eps_surprise_pct', 'eps_surprise_change',
]].sort_values('event_date', ascending=False).head(25))

## Modeling decision

The approved features use only announced values available by the anchor session. Missing values are meaningful: SEC filings have no direct surprise feature, and unmatched calls remain null rather than receiving a fabricated zero. The chronological model pipeline performs imputation inside each training fold.